Extract frames with OpenCV

In [2]:
import cv2
import math

video_path = r"C://Users//Avi//Desktop//Projectwork//Intelligent-Exploration-of-Educational-Videos//initial_test//STREAM_PROJECT_FAIRDI-1105FCDE.mp4"
cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
frame_interval_sec = 2  # one frame every 2 seconds
frame_interval = int(fps * frame_interval_sec)

frames = []  # list of (timestamp_sec, frame_image)

frame_idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_idx % frame_interval == 0:
        ts = frame_idx / fps
        frames.append((ts, frame))
    frame_idx += 1

cap.release()


In [ ]:
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from PIL import Image
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "Salesforce/blip2-opt-2.7b"  # or a smaller BLIP-2 variant
processor = Blip2Processor.from_pretrained(model_name)
model = Blip2ForConditionalGeneration.from_pretrained(
    model_name, torch_dtype=torch.float16 if device=="cuda" else torch.float32
).to(device)

def caption_frame(frame_bgr, prompt="Describe this frame in detail."):
    img_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(img_rgb)
    inputs = processor(pil_img, text=prompt, return_tensors="pt").to(device)
    out = model.generate(**inputs, max_new_tokens=40)
    caption = processor.decode(out[0], skip_special_tokens=True)
    return caption


c:\Users\Avi\Desktop\Projectwork\Intelligent-Exploration-of-Educational-Videos\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
`torch_dtype` is deprecated! Use `dtype` instead!
Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this

Loop over frames:

In [ ]:
captions = []
for ts, frame in frames:
    cap_text = caption_frame(frame)
    captions.append({"time": ts, "caption": cap_text})


Save captions

In [ ]:
import json

with open("frame_captions.json", "w", encoding="utf-8") as f:
    json.dump(captions, f, ensure_ascii=False, indent=2)
